# Topic: SQL Subquery Pattern (Scalar, Correlated, Derived, IN vs EXISTS)

## Definition (30-second explanation)
A subquery (or nested query) is a SQL query embedded within another SQL query, allowing you to break complex problems into smaller steps by using the result of one query as input to another. They can be placed in the `SELECT`, `FROM`, `WHERE`, and `HAVING` clauses. 

## Why Interviewers Ask This
Interviewers test subqueries to evaluate your ability to think relationally, handle complex filtering, and understand database performance implications. They specifically look for your awareness of common pitfalls like the "NULL trap" and knowing when to optimize a correlated subquery into a `JOIN` or Window Function.

## Core Concepts
*   **Scalar Subquery:** Returns exactly one value (1 row, 1 column) and is typically used in `SELECT` or `WHERE` clauses. Must use `LIMIT 1` if uniqueness isn't guaranteed.
*   **Correlated Subquery:** References columns from the outer query and executes once for every row processed by the outer query (O(n) executions).
*   **Derived Table:** A subquery located in the `FROM` clause that acts as a temporary table. It **must** be given an alias, even if unused.
*   **IN vs EXISTS:** `IN` filters by a list of values, while `EXISTS` evaluates to TRUE/FALSE if any matching record exists. 

## When to Use
*   Use a **Scalar Subquery** to compare individual row values against an aggregate metric (e.g., comparing an employee's salary to the company average).
*   Use a **Derived Table** to pre-aggregate or filter data before joining it to other large tables.
*   Use **EXISTS** to check for the presence of related records, especially in large datasets where short-circuiting improves performance.

## Advantages
*   Improves query readability by breaking complex logic into manageable, logical steps.
*   `EXISTS` and `NOT EXISTS` handle `NULL` values correctly and stop processing at the first match, making them highly efficient.
*   Derived tables can act similarly to CTEs, allowing complex multi-step transformations without creating actual views.

## Limitations
*   **Correlated Subqueries** can be extremely slow because they execute repeatedly for every row in the outer query.
*   Subqueries inside `IN` clauses can suffer from poor performance on very large lists compared to `EXISTS` or `JOIN`s.

## Common Comparisons
*   **WHERE IN vs EXISTS:** `EXISTS` is faster for large lists as it stops at the first match and handles NULLs safely; `IN` is slower and ignores NULLs.
*   **WHERE NOT IN vs NOT EXISTS:** `NOT EXISTS` is the safest way to find non-matching rows; `NOT IN` is a trap because if the subquery returns *any* NULL, the entire result is empty (always FALSE).
*   **Correlated Subquery vs Window Function:** Correlated subqueries run per row and are slow; Window functions compute aggregates over partitions simultaneously and are preferred for performance.

## Common Interview Traps
*   **The NOT IN + NULL Trap:** Writing `WHERE id NOT IN (SELECT id...)` without filtering NULLs in the subquery.
*   **Missing Derived Table Alias:** Forgetting to name a subquery in the `FROM` clause, which throws an immediate syntax error.
*   **Scalar Multi-Row Error:** Writing a scalar subquery that accidentally returns multiple rows, breaking the query.

## Python / SQL Syntax
```sql
-- Derived Table (Must have alias)
SELECT dept_stats.department, dept_stats.avg_salary
FROM (
    SELECT department, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
) AS dept_stats;

-- Correlated Subquery
SELECT emp_name, salary
FROM employees e1
WHERE salary > (
    SELECT AVG(e2.salary)
    FROM employees e2
    WHERE e2.department = e1.department
);
```

## 45-Second Interview Answer
Subqueries are queries nested within another query to simplify complex filtering or aggregation. A scalar subquery returns a single value, often used to compare a row against an aggregate. A derived table lives in the `FROM` clause to pre-aggregate data and must be aliased. The biggest interview pitfalls are correlated subqueries, which run once per row and can cause performance bottlenecks, and the `NOT IN` operator, which silently fails and returns nothing if the subquery contains a `NULL` value. I generally prefer `NOT EXISTS` or `LEFT JOIN` with a `NULL` check to safely avoid this trap.

## Example Questions:

### Q1: Find all customers who placed orders in January 2024 but NOT in February 2024.

**Ideal Interview Answer (MySQL):**
```sql
SELECT DISTINCT customer_id 
FROM orders 
WHERE order_date >= '2024-01-01' AND order_date < '2024-02-01'
  AND NOT EXISTS (
      SELECT 1 
      FROM orders o2 
      WHERE o2.customer_id = orders.customer_id 
        AND o2.order_date >= '2024-02-01' AND o2.order_date < '2024-03-01'
  );
```

**Common Mistakes Candidates Make:**
*   Using `NOT IN` without ensuring the subquery explicitly filters out `NULL` customer IDs.
*   Using functions on the indexed column (e.g., `MONTH(order_date) = 1`) which breaks SARGability (index usage).

**Likely Interviewer Follow-up:**
"How would you rewrite this exact query using a `LEFT JOIN` instead of `NOT EXISTS`?"

### Q2: Write a correlated subquery to find each product's price compared to its category's average price.

**Ideal Interview Answer (MySQL):**
```sql
SELECT 
    product_id, 
    price, 
    (SELECT AVG(price) 
     FROM products p2 
     WHERE p2.category_id = p1.category_id) AS category_avg_price
FROM products p1;
```

**Common Mistakes Candidates Make:**
*   Forgetting to alias the tables (`p1`, `p2`) to establish the correlation correctly.
*   Accidentally returning multiple columns or rows in the `SELECT` clause subquery, which throws a scalar subquery error.

**Likely Interviewer Follow-up:**
"This correlated subquery runs O(N) times. How can you rewrite this to be more performant using modern SQL features?" (Answer: Window Functions).

### Q3: Find the department with the highest average salary using only subqueries (no window functions).

**Ideal Interview Answer (MySQL):**
```sql
SELECT department
FROM (
    SELECT department, AVG(salary) AS avg_sal
    FROM employees
    GROUP BY department
) AS dept_avgs
ORDER BY avg_sal DESC
LIMIT 1;
```

**Common Mistakes Candidates Make:**
*   Trying to use `MAX(AVG(salary))`, which is invalid syntax in SQL.
*   Forgetting the alias `AS dept_avgs` for the derived table in the `FROM` clause.

**Likely Interviewer Follow-up:**
"If two departments are tied for the exact same highest average salary, this query only returns one. How would you modify it using subqueries to return *both* tied departments?"

### Q4: Convert this subquery into a CTE: SELECT * FROM (SELECT dept, AVG(salary) avg_sal FROM emp GROUP BY dept) t WHERE t.avg_sal > 70000.

**Ideal Interview Answer (MySQL):**
```sql
WITH DeptAverages AS (
    SELECT dept, AVG(salary) AS avg_sal
    FROM emp
    GROUP BY dept
)
SELECT * 
FROM DeptAverages 
WHERE avg_sal > 70000;
```

**Common Mistakes Candidates Make:**
*   Adding an alias to the CTE call in the main query like a derived table (e.g., `FROM DeptAverages t`), which isn't strictly necessary but sometimes causes confusion.
*   Forgetting the `AS` keyword after the CTE name.

**Likely Interviewer Follow-up:**
"In what scenario would a CTE be functionally superior to a derived table?" (Answer: When you need to reference the same aggregated temporary result multiple times in the main query).